In [11]:
from datasets import load_dataset, Dataset, concatenate_datasets, Image, DatasetDict
from PIL import Image as PILImage, ImageEnhance
import random
import io
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import bert_score



In [1]:
!gpustat

dgx01                     Thu Jan  8 18:02:35 2026  535.183.06
[0] NVIDIA A100-SXM4-80GB | 27°C,  ?? % |     2 / 81920 MB |
[1] NVIDIA A100-SXM4-80GB | 27°C,  ?? % |    87 / 81920 MB |
[2] NVIDIA A100-SXM4-80GB | 28°C,  ?? % |    87 / 81920 MB |
[3] NVIDIA A100-SXM4-80GB | 28°C,  ?? % |    87 / 81920 MB |
[4] NVIDIA A100-SXM4-80GB | 31°C,  ?? % |     2 / 81920 MB |
[5] NVIDIA A100-SXM4-80GB | 30°C,  ?? % |     2 / 81920 MB |
[6] NVIDIA A100-SXM4-80GB | 37°C,  ?? % |  7303 / 81920 MB | 66076055(7290M)
[7] NVIDIA A100-SXM4-80GB | 35°C,  ?? % |   503 / 81920 MB | praphan(490M)


In [13]:
# Load dataset
ds = load_dataset("Sarmistha/Final_idiom_all")
# keep only first 1000 samples
ds = DatasetDict({
    "train": ds["train"].select(range(10))
})

# cast image column
ds["train"] = ds["train"].cast_column("image", Image())
train_ds = ds["train"]
print(ds)
for i in range(5):
    sample = ds["train"][i]
    img = sample["image"]       
    actual = sample.get("Actual idiom")
    literal = sample.get("Literal Translation")
    english = sample.get("English Pronounciation")
    meaning = sample.get("Idiom meaning in English")   
    print(i, actual,literal,english,meaning, img.size)

DatasetDict({
    train: Dataset({
        features: ['image', 'Actual idiom', 'Literal Translation', 'English Pronounciation', 'Idiom meaning in English', 'Descriptive Meaning(Human Annotation)'],
        num_rows: 10
    })
})
0 अधजल गगरी छलकत जाय। A half-filled pitcher spills over. Adhjal Gagari Chalkat Jaye. Only ignorant men boast of their knowledge. (1024, 1024)
1 अपने मुँह मियाँ मिट्ठू बनाना। To praise oneself. Apne Munh Miya Mithu Banana. You should brag about yourself. (1024, 1024)
2 अपने हाथ में अपना भाग्य होना। To have one's fate in one's own hands. Apne Haath Mein Apna Bhagya Hona. to serve one's own interests (1024, 1024)
3 अपना उल्लू सीधा करना। To straighten one's own owl. Apna Ullu Seedha Karna. To serve one's own interests or deceive others for personal gain (1024, 1024)
4 अँगारे बरसना To rain embers. Angaare Barsna. Excessive heat. (1024, 1024)


In [14]:

# --- Augmentation Functions --- #
def change_contrast(img, factor=1.5):
    enhancer = ImageEnhance.Contrast(img)
    return enhancer.enhance(factor)

def random_rotation(img, max_angle=20):
    angle = random.uniform(-max_angle, max_angle)
    return img.rotate(angle, expand=True)

def skew(img, shear=10):
    w, h = img.size
    m = shear / 100.0
    return img.transform((w, h), PILImage.AFFINE, (1, m, 0, 0, 1, 0),
                         resample=PILImage.BICUBIC)

def flip(img):
    return img.transpose(PILImage.FLIP_LEFT_RIGHT)

# Convert PIL → bytes
def pil_to_bytes(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return {"bytes": buf.getvalue()}

# -------------------------------- #

augmented_rows = []
LIMIT = 3533
i = 0
print("Performing image augmentations...")
for sample in tqdm(train_ds, total=len(train_ds)):
    i+=1
    if i >= LIMIT:
        break
    img = sample["image"]

    # apply augmentations
    augs = [
        change_contrast(img),
        random_rotation(img),
        skew(img),
        flip(img)
    ]

    for aug_img in augs:
        augmented_rows.append({
            "image": pil_to_bytes(aug_img),
            "Actual idiom": sample["Actual idiom"],
            "Literal Translation": sample["Literal Translation"],
            "English Pronounciation": sample["English Pronounciation"],
            "Idiom meaning in English": sample["Idiom meaning in English"]
        })

print("\nConverting augmented rows to Dataset...")
augmented_ds = Dataset.from_list(augmented_rows).cast_column("image", Image())

print("\nConcatenating original and augmented datasets...")
final_ds = concatenate_datasets([train_ds, augmented_ds])

print("\nDone!")
print("Original samples:", len(train_ds))
print("Augmented samples:", len(augmented_ds))
print("Total samples:", len(final_ds))


Performing image augmentations...


100%|██████████| 10/10 [00:14<00:00,  1.50s/it]



Converting augmented rows to Dataset...

Concatenating original and augmented datasets...

Done!
Original samples: 10
Augmented samples: 40
Total samples: 50


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Actual SmolLM model on Hugging Face
feedback_name = "HuggingFaceTB/SmolLM3-3B"

feedback_tokenizer = AutoTokenizer.from_pretrained(feedback_name)
feedback_model = AutoModelForCausalLM.from_pretrained(
    feedback_name,
    device_map="auto",
    torch_dtype="auto"
)

feedback_model.eval()

def generate_feedback(real_meaning, predicted_meaning):
    prompt = (
        "Compare the REAL idiom meaning and the MODEL's prediction.\n\n"
        "REAL MEANING:\n"
        f"{real_meaning}\n\n"
        "MODEL PREDICTION:\n"
        f"{predicted_meaning}\n\n"
        "Task:\n"
        "- Analyze correctness\n"
        "- Identify mistakes or gaps\n"
        "- Give helpful improvement suggestions\n"
        "- Provide the feedback in 4–6 lines\n\n"
        "Begin feedback:\n"
    )

    inputs = feedback_tokenizer(
        prompt,
        return_tensors="pt"
    )
    inputs = {k: v.to(feedback_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = feedback_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=feedback_tokenizer.eos_token_id,
            pad_token_id=feedback_tokenizer.eos_token_id,
        )

    full_text = feedback_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Strip prompt from the generated text
    return full_text[len(prompt):].strip()




Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq, AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import torch

# ------------------- Model configuration -------------------
VLM_MODEL_NAME = "llava-hf/llava-1.5-7b-hf"
FEEDBACK_MODEL_NAME = "HuggingFaceTB/SmolLM3-3B"

# ------------------- Load VLM (LLaVA) -------------------
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_NAME)
vlm_model = AutoModelForVision2Seq.from_pretrained(
    VLM_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)
vlm_model.eval()

# ------------------- Load feedback model -------------------
feedback_tokenizer = AutoTokenizer.from_pretrained(FEEDBACK_MODEL_NAME)
feedback_model = AutoModelForCausalLM.from_pretrained(
    FEEDBACK_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)
feedback_model.eval()

# ------------------- Helper functions -------------------
def generate_idiom_output(
    image,  # kept only for API compatibility, not used
    literal_translation,
    english_pron,
    meaning,
    model=vlm_model,
    processor=vlm_processor
):
    """
    Pure text-only inference.
    Image is ignored completely.
    """

    prompt = (
        "You are an expert in interpreting idioms.\n"
        "Given the following data, give the meaning of the idiom.\n\n"
        f"Literal Translation: {literal_translation}\n"
        f"English Pronunciation: {english_pron}\n\n"
        "Give a short paragraph explanation."
    )

    # Optional: keep validation or remove entirely
    if not isinstance(image, Image.Image):
        raise ValueError("image must be a PIL.Image")

    inputs = processor(
        text=prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    response = processor.tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return response.strip()



def run_model_on_sample(sample, model=vlm_model, processor=vlm_processor):
    return generate_idiom_output(
        image=sample["image"],
        literal_translation=sample["Literal Translation"],
        english_pron=sample["English Pronounciation"],
        meaning=sample["Idiom meaning in English"],
        model=model,
        processor=processor
    )


def generate_feedback(
    real_meaning,
    predicted_meaning,
    model=feedback_model,
    tokenizer=feedback_tokenizer
):
    prompt = (
        "Compare the REAL idiom meaning and the MODEL's prediction.\n\n"
        "REAL MEANING:\n"
        f"{real_meaning}\n\n"
        "MODEL PREDICTION:\n"
        f"{predicted_meaning}\n\n"
        "Task:\n"
        "- Analyze correctness\n"
        "- Identify mistakes or gaps\n"
        "- Give helpful improvement suggestions\n"
        "- Provide feedback in 4–6 lines\n\n"
        "DO NOT COMMENT ON THE IMAGE. ONLY JUDGE THE TEXT.\n"
        "Begin feedback:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip()


# ------------------- Run predictions + feedback -------------------
def generate_feedback_map(
    dataset,
    vlm_model=vlm_model,
    vlm_processor=vlm_processor,
    feedback_model=feedback_model,
    feedback_tokenizer=feedback_tokenizer,
    N=50
):
    feedback_map = {}

    for i in range(N):
        sample = dataset[i]

        predicted = run_model_on_sample(sample, model=vlm_model, processor=vlm_processor)
        feedback = generate_feedback(
            sample["Idiom meaning in English"],
            predicted,
            model=feedback_model,
            tokenizer=feedback_tokenizer
        )

        feedback_map[f"img_{i}"] = {
            "actual": sample["Actual idiom"],
            "predicted_meaning": predicted,
            "real_meaning": sample["Idiom meaning in English"],
            "feedback": feedback
        }

    return feedback_map



/home/sarmistha/miniconda3/envs/suv_env/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2242: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel

# ------------------- Compute image embeddings -------------------
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).to("cuda")
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

def get_image_embedding(image):
    inputs = clip_processor(images=image, return_tensors="pt")

    # move inputs to GPU
    inputs = {k: v.to(clip_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)

    emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
    return emb

In [ ]:
print("Generating initial predictions + feedback...")

N = len(ds)
feedback_map = {}
image_embeddings = []

for i in tqdm(range(N), desc="Generating feedback + embeddings"):
    sample = augmented_ds[i]
    img = sample["image"]
    real_meaning = sample["Idiom meaning in English"]
    actual_idiom = sample["Actual idiom"]

    predicted = run_model_on_sample(sample)
    feedback = generate_feedback(real_meaning, predicted)

    feedback_map[f"img_{i}"] = {
        "actual": actual_idiom,
        "predicted_meaning": predicted,
        "real_meaning": real_meaning,
        "feedback": feedback
    }

    # store image embedding for similarity
    image_embeddings.append(get_image_embedding(img))

print("Initial feedback generation complete!\n")


In [ ]:
from transformers import AutoModel
from tqdm import tqdm
import numpy as np
import torch
import pandas as pd
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import bert_score
from sentence_transformers import SentenceTransformer

# ---------------- Sentence embedding setup ----------------
model_embed = SentenceTransformer('all-MiniLM-L6-v2')  # small, fast

def embed_sentence(text):
    return torch.tensor(model_embed.encode(text))

chencherry = SmoothingFunction()

print("Generating predictions using similar feedback and computing metrics...")

predictions_with_context = {}
metrics_map = {}
records = []  # store outputs for CSV

# accumulators for averages
metric_sums = {
    "R-1": 0.0, "R-2": 0.0, "R-L": 0.0,
    "B-1": 0.0, "B-2": 0.0, "B-3": 0.0, "B-L": 0.0,
    "BS": 0.0, "L1": 0.0, "L2": 0.0
}

N = len(augmented_ds)
for i in tqdm(range(N), desc="Processing images"):
    sample = augmented_ds[i]
    real_meaning = sample["Idiom meaning in English"]
    img = sample["image"]

    # -------- Similar image feedback --------
    if i > 0:
        curr_emb = get_image_embedding(img)
        sims = [
            torch.cosine_similarity(curr_emb, emb, dim=-1).item()
            for emb in image_embeddings[:i]
        ]
        most_sim_idx = sims.index(max(sims))
        similar_feedback = feedback_map[f"img_{most_sim_idx}"]["feedback"]
    else:
        similar_feedback = ""

    meaning_with_context = (
        f"{real_meaning}\n\nUse this previous feedback for context:\n{similar_feedback}"
        if similar_feedback else real_meaning
    )

    predicted_with_context = generate_idiom_output(
        img,
        sample["Actual idiom"],
        sample["Literal Translation"],
        meaning_with_context
    )

    predictions_with_context[f"img_{i}"] = predicted_with_context

    # -------- Save outputs for CSV --------
    records.append({
        "img_id": f"img_{i}",
        "actual_output": real_meaning,
        "predicted_output": predicted_with_context,
        "used_feedback": similar_feedback
    })

    # ---------------- Metrics ----------------
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
    )
    rouge_scores = scorer.score(real_meaning, predicted_with_context)

    R1 = rouge_scores['rouge1'].fmeasure
    R2 = rouge_scores['rouge2'].fmeasure
    RL = rouge_scores['rougeL'].fmeasure

    ref_tokens = [real_meaning.split()]
    pred_tokens = predicted_with_context.split()

    B1 = sentence_bleu(ref_tokens, pred_tokens, weights=(1, 0, 0, 0), smoothing_function=chencherry.method1)
    B2 = sentence_bleu(ref_tokens, pred_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=chencherry.method1)
    B3 = sentence_bleu(ref_tokens, pred_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=chencherry.method1)
    BL = sentence_bleu(ref_tokens, pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=chencherry.method1)

    P, R, F1 = bert_score.score(
        [predicted_with_context],
        [real_meaning],
        lang="en",
        rescale_with_baseline=True
    )
    BS = F1.item()

    emb_ref = embed_sentence(real_meaning)
    emb_pred = embed_sentence(predicted_with_context)

    L1 = torch.norm(emb_ref - emb_pred, p=1).item()
    L2 = torch.norm(emb_ref - emb_pred, p=2).item()

    metrics = {
        "R-1": R1, "R-2": R2, "R-L": RL,
        "B-1": B1, "B-2": B2, "B-3": B3, "B-L": BL,
        "BS": BS, "L1": L1, "L2": L2
    }

    metrics_map[f"img_{i}"] = metrics

    # accumulate
    for k in metric_sums:
        metric_sums[k] += metrics[k]

# ---------------- Final averages ----------------
avg_metrics = {k: v / N for k, v in metric_sums.items()}

print("\n===== FINAL AVERAGE METRICS =====")
for k, v in avg_metrics.items():
    print(f"{k}: {v:.4f}")

# ---------------- Save CSV ----------------
df = pd.DataFrame(records)
df.to_csv("actual_vs_predicted_outputs.csv", index=False)
print("\nSaved actual and predicted outputs to 'actual_vs_predicted_outputs.csv'")

print("\nPredictions with context and metrics computation complete!")




In [ ]:
import csv

with open("average_metrics.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Metric", "Value"])
    for k, v in avg_metrics.items():
        writer.writerow([k, v])

print("Average metrics saved to average_metrics.csv")

df = pd.DataFrame(records)
df.to_csv("predictions_and_metrics.csv", index=False)
print("\nSaved all predictions, actual outputs, and metrics to 'predictions_and_metrics.csv'")
